# Run on a KratosForge agent

Add this notebook to the environment's workspace with **LH_Meta** as its default lakehouse, and call it from a pipeline's Notebook activity. It writes a run request to `LH_Meta/Files/_fabric_desktop/requests/`, waits for the agent's status file, and fails the activity unless the run succeeded. See [Run from a Fabric pipeline](https://github.com/KratosDataGod/fabric-desktop/blob/main/docs/agent.md#run-from-a-fabric-pipeline). Preview: not yet run on a live tenant.


In [ ]:
# Parameters: set these from the pipeline's Notebook activity (base parameters).
env = "prod"                 # the environment to run; this notebook's default lakehouse is its LH_Meta
entities = ""                # comma-separated, e.g. "silver.orders,gold.fact_sales"; empty runs every entity
request_id = ""              # pass @pipeline().RunId, so a retried activity doesn't ask twice
submitted_by = ""            # e.g. @concat(pipeline().Pipeline, ' ', pipeline().RunId)
start_within_minutes = 60    # the request expires if the agent hasn't started it by then
timeout_minutes = 240        # stop waiting (and cancel it if it hasn't started) after this
poll_seconds = 30
control_root = "/lakehouse/default/Files/_fabric_desktop"


In [ ]:
"""Ask a KratosForge agent to run an environment, and wait for the answer.

Contract 1 of the pipeline mailbox: see docs/agent.md#run-from-a-fabric-pipeline.
"""
import datetime as dt
import json
import os
import time
import uuid

FINAL = ("succeeded", "failed", "cancelled", "expired", "rejected")


def submit(root, request_id, request):
    """Write requests/<id>.json once. Writing it under a temporary name first means the agent
    never reads half a file; submitting the same id again changes nothing."""
    os.makedirs(f"{root}/requests", exist_ok=True)
    path = f"{root}/requests/{request_id}.json"
    if os.path.exists(path):
        return False
    tmp = f"{root}/requests/.{request_id}.tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(request, f)
    os.replace(tmp, path)
    return True


def read_status(root, request_id):
    try:
        with open(f"{root}/status/{request_id}.json", encoding="utf-8") as f:
            return json.load(f)
    except (FileNotFoundError, ValueError):
        return None  # not taken yet, or being rewritten


def cancel(root, request_id):
    os.makedirs(f"{root}/cancel", exist_ok=True)
    with open(f"{root}/cancel/{request_id}.json", "w", encoding="utf-8") as f:
        json.dump({"cancelled_at": dt.datetime.now(dt.timezone.utc).isoformat()}, f)


def wait(root, request_id, timeout_minutes, poll_seconds, sleep=time.sleep, clock=time.monotonic):
    """The final status. Past the timeout it cancels the request (a run already going finishes)
    and raises TimeoutError."""
    end = clock() + timeout_minutes * 60
    while True:
        status = read_status(root, request_id)
        if status and status.get("state") in FINAL:
            return status
        if clock() >= end:
            cancel(root, request_id)
            state = status.get("state") if status else "not taken by any agent"
            raise TimeoutError(f"Request {request_id} was still {state} after {timeout_minutes} minutes; cancelled it")
        sleep(poll_seconds)


def make_request(env, entities, start_within_minutes, submitted_by):
    deadline = dt.datetime.now(dt.timezone.utc) + dt.timedelta(minutes=start_within_minutes)
    return {"contract": 1, "env": env,
            "entities": [e.strip() for e in entities.split(",") if e.strip()],
            "deadline": deadline.isoformat(timespec="seconds").replace("+00:00", "Z"),
            "submitted_by": submitted_by[:200]}


In [ ]:
request_id = request_id or uuid.uuid4().hex
submit(control_root, request_id, make_request(env, entities, start_within_minutes, submitted_by))
status = wait(control_root, request_id, timeout_minutes, poll_seconds)
print(json.dumps(status, indent=2))
if status["state"] != "succeeded":
    raise RuntimeError(f"{status['state']}: {status.get('reason') or status.get('error') or ''}")
try:
    import notebookutils
    notebookutils.notebook.exit(json.dumps(status))  # the activity's output, for later activities
except ImportError:
    pass
